## Homework 10

In this homework, you are going to use and compare two different trackers (of your liking) and compare the results.
*************************************************************************************************************
This notebook demonstrates object tracking on a sample penguin video using OpenCV. It implements KCF, MIL and CSRT trackers, saves individual frame captures with colored bounding boxes, compiles the results into a single side-by-side comparison video, and enables in-notebook playback.

In [1]:
import cv2
import urllib.request
import os
from IPython.display import Video, display
from time import perf_counter

# Same initial steps - url to vidio and output directory for conclutions
VIDEO_URL = "https://pixabay.com/videos/download/video-303793_medium.mp4" 
VIDEO_PATH = "303793.mp4" # I`ve attached thst in archive
OUTPUT_DIR = "tracking_results" 
OUTPUT_VIDEO_PATH = "tracking.webm"
MAX_FRAMES = 15

# make output directory. 
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Download video
if not os.path.exists(VIDEO_PATH):
    print("Downloading...")
    urllib.request.urlretrieve(VIDEO_URL, VIDEO_PATH)
    print("Yep. Video catched!")

In [2]:
# Tracking and saving 
def run_tracker(tracker_type, color, name_prefix, init_bbox):
    # Initializating of selected tracker
    if tracker_type == 'KCF':
        tracker = cv2.TrackerKCF_create()
    elif tracker_type == 'CSRT':
        tracker = cv2.TrackerCSRT_create()
    elif tracker_type == 'MIL':
        tracker = cv2.TrackerMIL_create()
    else:
        raise ValueError("wrong type of tracker you texted")

    cap = cv2.VideoCapture(VIDEO_PATH)
    ret, frame = cap.read()
    if not ret:
        print("Can`t open video.")
        cap.release()
        return
    
    tracker.init(frame, init_bbox)
    frame_count = 0
    while frame_count < MAX_FRAMES:
        ret, frame = cap.read()
        if not ret:
            break
        start = perf_counter()    
        success, bbox = tracker.update(frame)
        elapsed = perf_counter() - start
        update_fps = 1.0 / elapsed if elapsed > 0 else 0.0
        if success:
            x, y, w, h = [int(v) for v in bbox]
            # Drawing
            cv2.rectangle(frame, (x, y), (x + w, y + h), color, 2)
            cv2.putText(frame, tracker_type, (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
            print(f"[{tracker_type}] Frame {frame_count + 1}: Bounding Box -> X:{x}, Y:{y}, W:{w}, H:{h}", 'Hz:', update_fps)
        else:
            print(f"[{tracker_type}] Frame {frame_count + 1}: LOST OBJECT", 'Hz:', update_fps)

        frame_name = f"{OUTPUT_DIR}/{name_prefix}_frame_{frame_count + 1}.jpg"
        cv2.imwrite(frame_name, frame)
        frame_count += 1

    cap.release()

In [3]:
# Run code to save results in outpu_dir 

initial_bbox = (450, 320, 150, 250) # bounding box: (x, y, w, h). First pinguin at left

print("\n--- START KCF (Red color) ---")
run_tracker('KCF', (0, 0, 255), 'kcf', init_bbox=initial_bbox)

print("\n--- START CSRT (Blue color) ---")
run_tracker('CSRT', (255, 0, 0), 'csrt', init_bbox=initial_bbox)

print("\n--- START MIL (Green color) ---")
run_tracker('MIL', (0, 255, 0), 'mil', init_bbox=initial_bbox)


--- START KCF (Red color) ---
[KCF] Frame 1: Bounding Box -> X:450, Y:320, W:150, H:250 Hz: 19.915877722231475
[KCF] Frame 2: Bounding Box -> X:444, Y:318, W:150, H:250 Hz: 30.742681382342347
[KCF] Frame 3: Bounding Box -> X:442, Y:318, W:150, H:250 Hz: 17.14670555217777
[KCF] Frame 4: Bounding Box -> X:436, Y:318, W:150, H:250 Hz: 48.09246758203057
[KCF] Frame 5: Bounding Box -> X:436, Y:316, W:150, H:250 Hz: 15.228338194038951
[KCF] Frame 6: Bounding Box -> X:436, Y:314, W:150, H:250 Hz: 50.32268667945441
[KCF] Frame 7: Bounding Box -> X:434, Y:316, W:150, H:250 Hz: 50.29777036410583
[KCF] Frame 8: Bounding Box -> X:432, Y:314, W:150, H:250 Hz: 13.677780139457523
[KCF] Frame 9: Bounding Box -> X:432, Y:316, W:150, H:250 Hz: 27.906929501127735
[KCF] Frame 10: Bounding Box -> X:436, Y:314, W:150, H:250 Hz: 47.94866498690953
[KCF] Frame 11: Bounding Box -> X:436, Y:316, W:150, H:250 Hz: 28.74909055144816
[KCF] Frame 12: Bounding Box -> X:438, Y:316, W:150, H:250 Hz: 38.90187931609504
[

In [4]:
#   EXTRA STEP : Make video

print("\n--- Wait video tracking.webm compiling ---")

cap = cv2.VideoCapture(VIDEO_PATH)
ret, frame = cap.read()
if ret:
    height, width, layers = frame.shape
    fourcc = cv2.VideoWriter_fourcc(*'VP80')
    video_writer = cv2.VideoWriter(OUTPUT_VIDEO_PATH, fourcc, 10.0, (width, height))

    # Create 3 trackers for simult. watching 
    tracker_kcf = cv2.TrackerKCF_create()
    tracker_csrt = cv2.TrackerCSRT_create()
    tracker_mil = cv2.TrackerCSRT_create()
    
    initial_bbox_kcf = (450, 320, 150, 250)
    initial_bbox_csrt = (450, 320, 148, 248)
    initial_bbox_mil = (450, 320, 152, 252)
    tracker_kcf.init(frame, initial_bbox_kcf)
    tracker_csrt.init(frame, initial_bbox_csrt)
    tracker_mil.init(frame, initial_bbox_mil)

    frame_count = 0
    while frame_count < MAX_FRAMES:
        ret, frame = cap.read()
        if not ret:
            break

        # update KCF (RED)
        success_kcf, bbox_kcf = tracker_kcf.update(frame)
        if success_kcf:
            x, y, w, h = [int(v) for v in bbox_kcf]
            cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 0, 255), 2)
            cv2.putText(frame, "KCF", (x, y - 25), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)

        # Update CSRT (BLUE)
        success_csrt, bbox_csrt = tracker_csrt.update(frame)
        if success_csrt:
            x, y, w, h = [int(v) for v in bbox_csrt]
            cv2.rectangle(frame, (x, y), (x + w, y + h), (255, 0, 0), 2)
            cv2.putText(frame, "CSRT", (x, y - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)

        # Update MIL (GREEN)
        success_mil, bbox_mil = tracker_mil.update(frame)
        if success_mil:
            x, y, w, h = [int(v) for v in bbox_mil]
            cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)
            cv2.putText(frame, "MIL", (x, y - 15), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

        video_writer.write(frame)
        frame_count += 1

    video_writer.release()
    cap.release()
    print(f"Saved as '{OUTPUT_VIDEO_PATH}'")

# Start video
print("\n--- Watching. Enjoy ---")
display(Video(OUTPUT_VIDEO_PATH, embed=True, width=640))


--- Wait video tracking.webm compiling ---


OpenCV: FFMPEG: tag 0x30385056/'VP80' is not supported with codec id 139 and format 'webm / WebM'


Saved as 'tracking.webm'

--- Watching. Enjoy ---


Ok. Now look at penguin paw

In [5]:

initial_bbox = (680, 1800, 220, 80) # bounding box: (x, y, w, h). First pinguin at left + his paw

print("\n--- START KCF (Red color) ---")
run_tracker('KCF', (0, 0, 255), 'kcf', init_bbox=initial_bbox)

print("\n--- START CSRT (Blue color) ---")
run_tracker('CSRT', (255, 0, 0), 'csrt', init_bbox=initial_bbox)

print("\n--- START MIL (Green color) ---")
run_tracker('MIL', (0, 255, 0), 'mil', init_bbox=initial_bbox)


--- START KCF (Red color) ---
[KCF] Frame 1: Bounding Box -> X:680, Y:1800, W:220, H:80 Hz: 33.95264198052589
[KCF] Frame 2: Bounding Box -> X:676, Y:1796, W:220, H:80 Hz: 37.482667065425936
[KCF] Frame 3: Bounding Box -> X:670, Y:1790, W:220, H:80 Hz: 14.633505037712544
[KCF] Frame 4: Bounding Box -> X:666, Y:1784, W:220, H:80 Hz: 35.311672119970424
[KCF] Frame 5: Bounding Box -> X:662, Y:1780, W:220, H:80 Hz: 28.710719856074437
[KCF] Frame 6: Bounding Box -> X:662, Y:1778, W:220, H:80 Hz: 50.612233411849154
[KCF] Frame 7: Bounding Box -> X:654, Y:1774, W:220, H:80 Hz: 39.20261413510144
[KCF] Frame 8: Bounding Box -> X:648, Y:1770, W:220, H:80 Hz: 33.65696185368379
[KCF] Frame 9: Bounding Box -> X:642, Y:1766, W:220, H:80 Hz: 14.546493007623049
[KCF] Frame 10: Bounding Box -> X:638, Y:1768, W:220, H:80 Hz: 10.051213041337839
[KCF] Frame 11: LOST OBJECT Hz: 44.97486398802522
[KCF] Frame 12: LOST OBJECT Hz: 45.77144824401437
[KCF] Frame 13: LOST OBJECT Hz: 58.26658657437153
[KCF] Frame

Few conclusions:
 - CSRT the best, but slow and change bounding box [bb] size
 - MIL like CSRT, but more slow and doesn`t change [bb]
 - KCF fast, but lost [bb] if it changed shape. No change [bb]